# First model run with Tiny Aya

This notebook loads a small instruction-following model and asks it one benign question. Inference means using a model to generate an answer. Broader safety conclusions require a defined evaluation set and scoring method.

Before running, accept the access conditions for [`CohereLabs/tiny-aya-global`](https://huggingface.co/CohereLabs/tiny-aya-global) while signed in to Hugging Face. This gated model is about 3.35 billion parameters. The reported Colab T4 has 14.6 GB of memory, so this setup uses half precision and eager attention to reduce memory use. Actual compatibility will be checked when you run it.

Use a Colab GPU runtime. Run cells in order. Enable access to `HF_TOKEN` in this notebook through Colab Secrets.

## 1. Install the libraries

Colab already includes PyTorch. This cell installs Transformers and Accelerate for loading the model.

In [ ]:
%pip -q install 'transformers>=4.57,<5' 'accelerate>=1,<2'

## 2. Check the runtime and read the secret

The token is read from Colab Secrets to authenticate model access. The result record contains the prompt, response, and experiment metadata.

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import HfApi

assert torch.cuda.is_available(), "Select a GPU runtime in Colab, then rerun this cell."
hf_token = userdata.get("HF_TOKEN")
assert hf_token, "Add HF_TOKEN in Colab Secrets and enable notebook access."

model_id = "CohereLabs/tiny-aya-global"
model_info = HfApi(token=hf_token).model_info(model_id)
model_revision = model_info.sha
assert model_revision, "Could not resolve a model revision. Check Hugging Face access and retry."
print(f"GPU ready: {torch.cuda.get_device_name(0)}")
print(f"Model revision resolved: {model_revision}")

## 3. Load the model

We pin both downloads to the same resolved revision so the recorded run identifies the model snapshot. The official chat template formats the messages expected by the model.

In [ ]:
import time
from datetime import datetime, timezone
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    model_id, revision=model_revision, token=hf_token
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    revision=model_revision,
    token=hf_token,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    attn_implementation="eager",
)
model.eval()
print("Tokenizer and model loaded.")

## 4. Predict, then ask the model

Before running the next cell, write down what you expect a responsible answer to say. Then compare that prediction with the model's response.
The displayed answer contains the newly generated text, with formatting tokens removed.


In [ ]:
messages = [
    {"role": "user", "content": "Explain phishing in two sentences and give one way to protect yourself."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)

settings = {"do_sample": False, "max_new_tokens": 120}
torch.cuda.synchronize()
start = time.perf_counter()
with torch.inference_mode():
    generated = model.generate(**inputs, **settings)
torch.cuda.synchronize()
latency_seconds = time.perf_counter() - start

new_tokens = generated[0, inputs["input_ids"].shape[-1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
print(answer)
print(f"\nGeneration time: {latency_seconds:.2f} seconds")

## 5. Save a run record

The JSON file records the model revision, software and runtime details, prompt, generation settings, answer, and elapsed generation time. The file is saved in the temporary Colab session at `/content/results/first_run.json`.

In [ ]:
import json
import platform
from pathlib import Path
import accelerate
import transformers

record = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": model_id,
    "model_revision": model_revision,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "accelerate_version": accelerate.__version__,
    "cuda_version": torch.version.cuda,
    "gpu_name": torch.cuda.get_device_name(0),
    "messages": messages,
    "generation_settings": settings,
    "answer": answer,
    "generation_latency_seconds": round(latency_seconds, 3),
}

results_dir = Path("/content/results")
results_dir.mkdir(parents=True, exist_ok=True)
result_path = results_dir / "first_run.json"
result_path.write_text(json.dumps(record, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"Saved run record to {result_path}")

## Download the result

Run this cell to keep a copy on your computer. The Colab session's `/content` files are temporary.

In [ ]:
from google.colab import files
files.download(str(result_path))

## Reflect on the answer

- Did the response match your prediction?
- Was the explanation accurate and easy to understand?
- Did the protection step give a useful action?
- What would you want to check before drawing a broader conclusion?

Keep these observations with the saved result. We will use them to plan a larger experiment.